# Tuned for 20s Horizon — Summary of Changes

This is a modified version of `cross_device_lstm_forecast_20.ipynb`. **The block-building, phase-grouping, and train/val/test split logic (Parts 6-10) are untouched** — only the following were changed:

1. **Lookback window: 60s -> 40s.** Shorter, more relevant history for a 20s-ahead target; fewer timesteps means less overfitting risk.
2. **Data cleaning (`clean_run`)**: drop physically implausible temperature/usage readings; winsorize other features to 1st-99th percentile instead of dropping rows (preserves time continuity).
3. **Target cleaning**: winsorize the *training* dT target to the 1st-99th percentile only (val/test left untouched for honest evaluation).
4. **Regularization**: `recurrent_dropout=0.1` on the LSTM, L2 weight decay on LSTM + Dense, Dropout 0.20 -> 0.30.
5. **Hyperparameters**: LSTM units 32 -> 40, learning rate 0.0005 -> 0.0008 with `clipnorm=1.0`, batch size 64 -> 32, early-stopping patience 12 -> 10.
6. **Bug fix**: `checkpoint` (ModelCheckpoint) was referenced in `model.fit()` but never defined anywhere — this would have crashed training. It's now properly created.
7. **Bug fix**: three cells after the best-model evaluation were silently re-predicting with `model` (last-epoch weights) instead of `best_model` (best checkpoint), overwriting the correct results. Removed.

**Note:** The source CSVs (`manan_merged_experiment.csv`, `prabh_merged_experiment.csv`) weren't available in this session, so these changes are reasoned from the code/config rather than empirically verified. Run this end-to-end on your data and compare `internal_lstm` MAE/RMSE against your original run's numbers. If 40s underperforms, the sweep-worthy alternatives are 30s and 50s — everything downstream (Part 12 onward) will pick up a new `LOOKBACK_SECONDS` automatically.

# Part 1: Imports and Configuration

In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout)

from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)

import joblib
import random

In [117]:
SEED = 42
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

Configuration

In [119]:
# Approximate Sensor interval
SAMPLE_INTERVAL_SECONDS = 2

# TUNED: shortened from 60s to 40s.
# For a short 20s forecast horizon, a 60s (30-sample) lookback carries a lot of
# older, less-relevant history that mostly adds noise/variance to the LSTM's
# hidden state. A 40s (20-sample) window still captures the recent
# heating/cooling trend (slope) needed to predict 20s ahead, while giving the
# model fewer, more relevant timesteps to learn from -> less overfitting and
# faster training.
LOOKBACK_SECONDS = 40

# Predict 20 seconds into the future
FORECAST_SECONDS = 20

LOOKBACK = LOOKBACK_SECONDS // SAMPLE_INTERVAL_SECONDS
HORIZON = FORECAST_SECONDS // SAMPLE_INTERVAL_SECONDS

print("Lookback samples: ", LOOKBACK)
print("Forecast Horizon samples: ", HORIZON)


Lookback samples:  20
Forecast Horizon samples:  10


# Part 2: Loading Datasets

In [ ]:
FILES = {
    "manan": "manan_merged_experiment.csv",
    "prabhsimrat": "prabh_merged_experiment.csv",
    "mukul": "mukul_merged_experiment.csv",
    "prabh2":"prabh2_merged_experiment.csv"
}

In [123]:
datasets ={}

for name, path in FILES.items():
    df = pd.read_csv(path)
    df["timestamp"]= pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop= True)
    df["source"] = name
    datasets[name]= df
    print(name, "-", df.shape)

manan - (14803, 13)
prabhsimrat - (15365, 13)
mukul - (18890, 13)
prabh2 - (14513, 13)


In [125]:
FEATURE_COLS = [
    "cpuUsage",
    "ramUsage",
    "networkConnections",
    "processCount",
    "cpuPackagePower",
    "cpuTemperature"
]

In [127]:
for name, df in datasets.items():

    print(f"\n===== {name} =====")

    print(
        df[
            [
                "cpuUsage",
                "cpuPackagePower",
                "cpuAverageClock",
                "cpuTemperature"
            ]
        ].describe().loc[
            ["min", "mean", "std", "max"]
        ]
    )


===== manan =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     9.200000         9.600000       403.200000       60.000000
mean   58.584071        40.438654      2419.553636       86.093292
std    30.660576        15.644110       471.266927       11.716394
max   100.000000        66.600000      2973.600000       99.000000

===== prabhsimrat =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     1.000000         6.000000      2810.000000       64.100000
mean   49.854247        29.637592      3617.252652       87.949492
std    34.927009        11.036970       287.802921       10.251866
max   100.000000        50.900000      4492.000000       95.900000

===== mukul =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     0.100000         2.200000       179.000000       43.300000
mean   40.209010        11.000021      3082.978507       64.240053
std    35.919177         6.123213       513.787701       12.145046

# Part 3: Cleaning 

After running (after_stress_generator.ipynb)- most of the cleaning is already done

In [131]:
def clean_run(df):
    df = df.copy()
    # Keep only valid sensor data
    df = df.dropna(subset=FEATURE_COLS)

    # ---- TUNED: additional data cleaning ----
    # Drop physically implausible sensor glitches. A single bad reading (e.g. a
    # negative/zero temperature or >100% usage spike) can inject a large,
    # meaningless dT into the target and encourage the model to chase noise.
    df = df[(df["cpuTemperature"] > 0) & (df["cpuTemperature"] < 110)]
    df = df[(df["cpuUsage"] >= 0) & (df["cpuUsage"] <= 100)]

    # Winsorize (clip, don't drop) the remaining noisy features to their
    # 1st-99th percentile. Clipping instead of dropping preserves the
    # contiguous time index that the sequence builder relies on.
    for col in ["ramUsage", "networkConnections", "processCount", "cpuPackagePower"]:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    df = df.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    return df


In [133]:
for name in datasets:

    datasets[name] = clean_run(
        datasets[name]
    )

    print(
        name,
        len(datasets[name])
    )

manan 14803
prabhsimrat 15365
mukul 18890
prabh2 14513


# Checking Sampling Gaps

In [136]:
def inspect_time_gaps(df, name):

    gaps = (
        df["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    print(f"\n{name}")

    print(
        "Median interval:",
        gaps.median()
    )

    print(
        "Maximum gap:",
        gaps.max()
    )

    print(
        "Gaps > 5 seconds:",
        (gaps > 5).sum()
    )

In [138]:
for name, df in datasets.items():

    inspect_time_gaps(
        df,
        name
    )


manan
Median interval: 2.0298681
Maximum gap: 4.3050749
Gaps > 5 seconds: 0

prabhsimrat
Median interval: 2.02355955
Maximum gap: 7.0642233
Gaps > 5 seconds: 1

mukul
Median interval: 2.0282471
Maximum gap: 5.3255022
Gaps > 5 seconds: 1

prabh2
Median interval: 2.0220667
Maximum gap: 4.3147634
Gaps > 5 seconds: 0


# Part 6: Create Balanced Time Blocks 

In [141]:
TRAIN_RUNS = ["manan", "prabhsimrat","mukul","prabh2"]
TEST_RUNS = ["manan", "prabhsimrat","mukul","prabh2"]

In [143]:
BLOCK_MINUTES = 10

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

In [145]:
def get_phase_group(mode):

    mode = str(mode).upper()

    if "COOLING" in mode:
        return "COOLING"

    if mode in [
        "INITIAL_IDLE",
        "PRE_EXPERIMENT",
        "POST_EXPERIMENT"
    ]:
        return "IDLE"

    if mode in [
        "RAMP",
        "CHAOS",
        "TRANSITION",
        "MIXED"
    ]:
        return mode

    return "OTHER"

In [147]:
for name, df in datasets.items():

    df = df.copy()

    df["phaseGroup"] = (
        df["mode"]
        .apply(get_phase_group)
    )

    datasets[name] = df

    print(f"\n===== {name} =====")

    print(
        df["phaseGroup"]
        .value_counts()
    )


===== manan =====
phaseGroup
MIXED         5298
CHAOS         2658
RAMP          2656
TRANSITION    2654
OTHER          739
COOLING        739
IDLE            59
Name: count, dtype: int64

===== prabhsimrat =====
phaseGroup
MIXED         5306
CHAOS         2669
RAMP          2664
TRANSITION    2656
OTHER         1574
COOLING        445
IDLE            51
Name: count, dtype: int64

===== mukul =====
phaseGroup
MIXED         5289
OTHER         4873
RAMP          2650
CHAOS         2650
TRANSITION    2646
COOLING        737
IDLE            45
Name: count, dtype: int64

===== prabh2 =====
phaseGroup
MIXED         5323
CHAOS         2669
RAMP          2665
TRANSITION    2658
COOLING        742
OTHER          420
IDLE            36
Name: count, dtype: int64


Now Creating Contiguous Blocks

In [150]:
def create_time_blocks(df, run_name, block_minutes = 10):
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop= True)
    all_blocks = []
    block_counter = 0
    
    # --------------------- Processing Each Contiguous Phase Separately----------------------------
    
    phase_change = (df["phaseGroup"]!=df["phaseGroup"].shift())
    
    df["phaseSegment"]= (phase_change.cumsum())
    
    for segment_id, segment in df.groupby("phaseSegment"):
        segment = (segment.sort_values("timestamp").copy())
        phase = (segment["phaseGroup"].iloc[0])
        segment_start = (segment["timestamp"].min())
        
        # Assign 10 minute block number
        elapsed_minutes = (segment["timestamp"] - segment_start).dt.total_seconds() / 60
        
        segment["localBlock"] = (elapsed_minutes // block_minutes).astype(int)
        
        for local_block, block in segment.groupby("localBlock"):
            block = block.copy()
            block["blockId"] = (f"{run_name}"
                                f"{block_counter}")
            block["runName"] = run_name
            all_blocks.append(block)
            block_counter+=1
    
    return all_blocks

In [152]:
all_blocks = []

for name in TRAIN_RUNS:

    run_blocks = create_time_blocks(

        df=datasets[name],

        run_name=name,

        block_minutes=BLOCK_MINUTES
    )

    all_blocks.extend(
        run_blocks
    )

    print(
        name,
        "blocks:",
        len(run_blocks)
    )

manan blocks: 54
prabhsimrat blocks: 55
mukul blocks: 67
prabh2 blocks: 52


# 7. Remove Blocks that are too short

In [155]:
MIN_REQUIRED_SAMPLES = (
    LOOKBACK
    + HORIZON
    + 5
)

In [157]:
valid_blocks = []

removed_blocks = []

for block in all_blocks:

    if len(block) >= MIN_REQUIRED_SAMPLES:

        valid_blocks.append(block)

    else:

        removed_blocks.append(block)


print(
    "Valid blocks:",
    len(valid_blocks)
)

print(
    "Removed short blocks:",
    len(removed_blocks)
)

Valid blocks: 227
Removed short blocks: 1


In [159]:
block_summary = []

for block in valid_blocks:

    block_summary.append({

        "blockId":
            block["blockId"].iloc[0],

        "runName":
            block["runName"].iloc[0],

        "phaseGroup":
            block["phaseGroup"].iloc[0],

        "samples":
            len(block),

        "startTime":
            block["timestamp"].min(),

        "endTime":
            block["timestamp"].max()
    })


block_summary = pd.DataFrame(
    block_summary
)

display(
    block_summary.head(20)
)

,blockId,runName,phaseGroup,samples,startTime,endTime
0,manan1,manan,IDLE,59,2026-07-06 02:15:52.811050700+05:30,2026-07-06 02:17:50.805294900+05:30
1,manan2,manan,RAMP,296,2026-07-06 02:17:52.834077200+05:30,2026-07-06 02:27:51.946574700+05:30
2,manan3,manan,RAMP,295,2026-07-06 02:27:53.976796400+05:30,2026-07-06 02:37:52.628830900+05:30
3,manan4,manan,RAMP,295,2026-07-06 02:37:54.652708500+05:30,2026-07-06 02:47:51.481890400+05:30
4,manan5,manan,RAMP,296,2026-07-06 02:47:53.518745400+05:30,2026-07-06 02:57:51.959968500+05:30
5,manan6,manan,RAMP,292,2026-07-06 02:57:54.002098900+05:30,2026-07-06 03:07:50.885778200+05:30
6,manan7,manan,RAMP,296,2026-07-06 03:07:52.939946600+05:30,2026-07-06 03:17:51.375515200+05:30
7,manan8,manan,RAMP,296,2026-07-06 03:17:53.410543100+05:30,2026-07-06 03:27:52.352622600+05:30
8,manan9,manan,RAMP,295,2026-07-06 03:27:54.375094400+05:30,2026-07-06 03:37:52.734991300+05:30
9,manan10,manan,RAMP,295,2026-07-06 03:37:54.764023700+05:30,2026-07-06 03:47:51.479478400+05:30


In [161]:
print(
    block_summary[
        "phaseGroup"
    ].value_counts()
)

phaseGroup
MIXED         72
RAMP          36
CHAOS         36
TRANSITION    36
OTHER         28
COOLING       15
IDLE           4
Name: count, dtype: int64


# 8. Split whole blocks into train, validation and internal test

In [164]:
from sklearn.model_selection import train_test_split

In [166]:
def split_blocks_balanced(
    block_summary,
    seed=42
):

    train_ids = []
    val_ids = []
    test_ids = []

    # ----------------------------------------
    # Split separately by:
    # device + experiment phase
    # ----------------------------------------

    grouped = block_summary.groupby(
        [
            "runName",
            "phaseGroup"
        ]
    )


    for (
        run_name,
        phase
    ), group in grouped:

        ids = (
            group["blockId"]
            .tolist()
        )


        # Reproducible random order
        rng = np.random.default_rng(
            seed
        )

        rng.shuffle(ids)


        n = len(ids)


        print(
            run_name,
            phase,
            "blocks:",
            n
        )


        # ------------------------------------
        # Very small groups
        # ------------------------------------

        if n == 1:

            train_ids.extend(ids)

            continue


        if n == 2:

            train_ids.append(ids[0])
            val_ids.append(ids[1])

            continue


        # ------------------------------------
        # Normal groups
        # ------------------------------------

        n_train = max(
            1,
            int(round(n * TRAIN_RATIO))
        )

        n_val = max(
            1,
            int(round(n * VAL_RATIO))
        )


        # Ensure at least one test block
        if (
            n_train
            + n_val
            >= n
        ):

            n_train = n - 2
            n_val = 1
            

        train_ids.extend(
            ids[:n_train]
        )


        val_ids.extend(
            ids[
                n_train:
                n_train + n_val
            ]
        )


        test_ids.extend(
            ids[
                n_train + n_val:
            ]
        )


    return (
        train_ids,
        val_ids,
        test_ids
    )

In [168]:
(
    train_block_ids,
    val_block_ids,
    test_block_ids

) = split_blocks_balanced(
    block_summary,
    seed=SEED
)

manan CHAOS blocks: 9
manan COOLING blocks: 4
manan IDLE blocks: 1
manan MIXED blocks: 18
manan OTHER blocks: 3
manan RAMP blocks: 9
manan TRANSITION blocks: 9
mukul CHAOS blocks: 9
mukul COOLING blocks: 4
mukul IDLE blocks: 1
mukul MIXED blocks: 18
mukul OTHER blocks: 17
mukul RAMP blocks: 9
mukul TRANSITION blocks: 9
prabh2 CHAOS blocks: 9
prabh2 COOLING blocks: 4
prabh2 IDLE blocks: 1
prabh2 MIXED blocks: 18
prabh2 OTHER blocks: 2
prabh2 RAMP blocks: 9
prabh2 TRANSITION blocks: 9
prabhsimrat CHAOS blocks: 9
prabhsimrat COOLING blocks: 3
prabhsimrat IDLE blocks: 1
prabhsimrat MIXED blocks: 18
prabhsimrat OTHER blocks: 6
prabhsimrat RAMP blocks: 9
prabhsimrat TRANSITION blocks: 9


In [170]:
print(
    "\nTraining blocks:",
    len(train_block_ids)
)

print(
    "Validation blocks:",
    len(val_block_ids)
)

print(
    "Internal test blocks:",
    len(test_block_ids)
)


Training blocks: 153
Validation blocks: 34
Internal test blocks: 40


In [172]:
block_summary["split"] = (
    "UNASSIGNED"
)

block_summary.loc[

    block_summary["blockId"]
    .isin(train_block_ids),

    "split"

] = "TRAIN"


block_summary.loc[

    block_summary["blockId"]
    .isin(val_block_ids),

    "split"

] = "VALIDATION"


block_summary.loc[

    block_summary["blockId"]
    .isin(test_block_ids),

    "split"

] = "TEST"

In [174]:
coverage = pd.crosstab(

    block_summary["phaseGroup"],

    block_summary["split"]
)

display(coverage)

split,TEST,TRAIN,VALIDATION
phaseGroup,,,
CHAOS,8,24,4
COOLING,4,7,4
IDLE,0,4,0
MIXED,8,52,12
OTHER,4,18,6
RAMP,8,24,4
TRANSITION,8,24,4


In [176]:
device_coverage = pd.crosstab(

    block_summary["runName"],

    block_summary["split"]
)

display(device_coverage)

split,TEST,TRAIN,VALIDATION
runName,,,
manan,10,35,8
mukul,11,46,10
prabh2,9,35,8
prabhsimrat,10,37,8


# 10. Converting IDs back to block lists

In [179]:
train_blocks = []

val_blocks = []

test_blocks = []


for block in valid_blocks:

    block_id = (
        block["blockId"]
        .iloc[0]
    )


    if block_id in train_block_ids:

        train_blocks.append(
            block
        )


    elif block_id in val_block_ids:

        val_blocks.append(
            block
        )


    elif block_id in test_block_ids:

        test_blocks.append(
            block
        )

In [181]:
print(
    "Train blocks:",
    len(train_blocks)
)

print(
    "Validation blocks:",
    len(val_blocks)
)

print(
    "Internal test blocks:",
    len(test_blocks)
)

Train blocks: 153
Validation blocks: 34
Internal test blocks: 40


# Part 11: Fitting the scaler on traininig blocks only

In [184]:
scaler_training_data = pd.concat(

    [
        block[FEATURE_COLS]
        for block in train_blocks
    ],

    ignore_index=True
)

In [186]:
scaler_X = StandardScaler()

scaler_X.fit(
    scaler_training_data
)

StandardScaler()

In [188]:
joblib.dump(

    scaler_X,

    "cross_device_feature_scaler.pkl"
)

print(
    "Scaler fitted only on training blocks."
)

Scaler fitted only on training blocks.


# Part 12: Create sequences inside each block

In [191]:
def create_sequences_from_block(
    block,
    scaler,
    lookback,
    horizon,
    max_gap_seconds=5
):

    block = (
        block
        .sort_values("timestamp")
        .reset_index(drop=True)
        .copy()
    )


    scaled_features = scaler.transform(
        block[FEATURE_COLS]
    )


    temperatures = (

        block["cpuTemperature"]

        .to_numpy()
    )


    timestamps = (

        block["timestamp"]

        .to_numpy()
    )


    X = []
    y = []

    current_temperatures = []

    future_temperatures = []

    prediction_timestamps = []


    for i in range(

        lookback,

        len(block) - horizon + 1
    ):


        sequence_start = (
            i - lookback
        )


        current_index = (
            i - 1
        )


        future_index = (

            current_index

            + horizon
        )


        if future_index >= len(block):

            break


        # ------------------------------------
        # Check entire history → future period
        # ------------------------------------

        relevant_times = (

            block["timestamp"]

            .iloc[
                sequence_start:
                future_index + 1
            ]
        )


        gaps = (

            relevant_times

            .diff()

            .dt.total_seconds()

            .dropna()
        )


        if (
            gaps > max_gap_seconds
        ).any():

            continue


        # ------------------------------------
        # Input
        # ------------------------------------

        X.append(

            scaled_features[
                sequence_start:i
            ]
        )


        # ------------------------------------
        # Target
        # ------------------------------------

        current_temp = (

            temperatures[
                current_index
            ]
        )


        future_temp = (

            temperatures[
                future_index
            ]
        )


        delta_temp = (

            future_temp

            - current_temp
        )


        y.append(
            delta_temp
        )


        current_temperatures.append(
            current_temp
        )


        future_temperatures.append(
            future_temp
        )


        prediction_timestamps.append(
            timestamps[
                future_index
            ]
        )


    return (

        np.asarray(X),

        np.asarray(y),

        np.asarray(
            current_temperatures
        ),

        np.asarray(
            future_temperatures
        ),

        np.asarray(
            prediction_timestamps
        )
    )

# 13. Create datasets from block lists

In [194]:
def create_dataset_from_blocks(
    blocks,
    scaler
):

    all_X = []
    all_y = []

    all_current = []
    all_future = []

    all_timestamps = []

    all_phases = []
    all_runs = []


    for block in blocks:


        (
            X,
            y,
            current,
            future,
            timestamps

        ) = create_sequences_from_block(

            block=block,

            scaler=scaler,

            lookback=LOOKBACK,

            horizon=HORIZON
        )


        if len(X) == 0:

            continue


        phase = (
            block[
                "phaseGroup"
            ].iloc[0]
        )


        run_name = (
            block[
                "runName"
            ].iloc[0]
        )


        all_X.append(X)

        all_y.append(y)

        all_current.append(
            current
        )

        all_future.append(
            future
        )

        all_timestamps.append(
            timestamps
        )


        all_phases.extend(

            [phase] * len(X)
        )


        all_runs.extend(

            [run_name] * len(X)
        )


    return (

        np.concatenate(all_X),

        np.concatenate(all_y),

        np.concatenate(all_current),

        np.concatenate(all_future),

        np.concatenate(all_timestamps),

        np.asarray(all_phases),

        np.asarray(all_runs)
    )

In [196]:
(
    X_train,
    y_train,
    train_current,
    train_actual,
    train_timestamps,
    train_phases,
    train_runs

) = create_dataset_from_blocks(

    train_blocks,

    scaler_X
)

In [198]:
# TUNED: data cleaning on the training target only.
# Extreme dT values are almost always transient sensor glitches rather than
# real 20s temperature swings. We winsorize the TRAINING target to the
# 1st-99th percentile so the model isn't pulled toward noise, while leaving
# validation/test targets untouched so evaluation metrics stay honest.
y_train_lower = np.percentile(y_train, 1)
y_train_upper = np.percentile(y_train, 99)

print(f"Clipping training dT targets to [{y_train_lower:.3f}, {y_train_upper:.3f}] deg C (1st-99th pct)")

y_train = np.clip(y_train, y_train_lower, y_train_upper)


Clipping training dT targets to [-20.700, 20.800] deg C (1st-99th pct)


In [200]:
(
    X_val,
    y_val,
    val_current,
    val_actual,
    val_timestamps,
    val_phases,
    val_runs

) = create_dataset_from_blocks(

    val_blocks,

    scaler_X
)

In [202]:
(
    X_test,
    y_test,
    test_current,
    test_actual,
    test_timestamps,
    test_phases,
    test_runs

) = create_dataset_from_blocks(

    test_blocks,

    scaler_X
)

In [204]:
print("\nTRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nVALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nINTERNAL TEST")
print("X:", X_test.shape)
print("y:", y_test.shape)


TRAIN
X: (38396, 20, 6)
y: (38396,)

VALIDATION
X: (8470, 20, 6)
y: (8470,)

INTERNAL TEST
X: (10059, 20, 6)
y: (10059,)


# 14. Verify Actual Sequence Diversity

In [207]:
def show_sequence_distribution(
    phases,
    name
):

    print(
        f"\n===== {name} ====="
    )

    print(

        pd.Series(phases)

        .value_counts()
    )

In [209]:
show_sequence_distribution(
    train_phases,
    "TRAIN"
)

show_sequence_distribution(
    val_phases,
    "VALIDATION"
)

show_sequence_distribution(
    test_phases,
    "INTERNAL TEST"
)


===== TRAIN =====
MIXED         13807
CHAOS          6404
RAMP           6397
TRANSITION     6350
OTHER          4117
COOLING        1246
IDLE             75
Name: count, dtype: int64

===== VALIDATION =====
MIXED         3197
OTHER         1603
TRANSITION    1066
CHAOS         1065
RAMP          1062
COOLING        477
Name: count, dtype: int64

===== INTERNAL TEST =====
CHAOS         2133
RAMP          2132
TRANSITION    2125
MIXED         2124
OTHER         1069
COOLING        476
Name: count, dtype: int64


# 15. Building the LSTM

In [212]:
# TUNED architecture:
# - LSTM units 32 -> 40 (a small capacity bump to compensate for the shorter
#   lookback, still far from the "complex model" territory that underperformed)
# - recurrent_dropout=0.1 added: regularizes the recurrent connections
#   specifically, which plain Dropout on the output doesn't touch
# - L2 weight regularization added to LSTM + Dense to discourage large weights
#   and reduce overfitting
# - Dropout 0.20 -> 0.30: slightly stronger regularization after the LSTM
model = Sequential([
    Input(shape=(LOOKBACK, len(FEATURE_COLS))),

    LSTM(
        40,
        recurrent_dropout=0.1,
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),

    Dropout(0.30),

    Dense(16, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),

    Dense(1)
])


In [214]:
# TUNED: learning rate nudged up slightly (0.0005 -> 0.0008) since the
# shorter lookback + added regularization mean the model needs a bit more
# step size to converge in the same number of epochs. clipnorm=1.0 added for
# training stability (guards against occasional large gradients from any
# remaining outliers).
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0008,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.Huber(delta=2.0),
    metrics=["mae"]
)


# Part 16: Train

In [217]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=0.01,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# FIX: `checkpoint` was referenced in model.fit(...) below but was never
# actually created anywhere in the notebook (only ModelCheckpoint was
# imported). Without this, model.fit() would raise a NameError.
checkpoint = ModelCheckpoint(
    "best_cross_device_lstm.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)


Train

In [220]:
# TUNED: batch_size 64 -> 32. Smaller batches give noisier, more frequent
# gradient updates, which for this dataset size tends to generalize slightly
# better than large batches (a mild regularizing effect) at some cost to
# training speed.
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=32,

    shuffle=True,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)


Epoch 1/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.6804 - mae: 2.4299
Epoch 1: val_loss improved from inf to 2.75063, saving model to best_cross_device_lstm.keras
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 3.6804 - mae: 2.4299 - val_loss: 2.7506 - val_mae: 1.9037 - learning_rate: 8.0000e-04
Epoch 2/100
1192/1200 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.6026 - mae: 2.3865
Epoch 2: val_loss improved from 2.75063 to 2.74167, saving model to best_cross_device_lstm.keras
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 3.6025 - mae: 2.3864 - val_loss: 2.7417 - val_mae: 1.8994 - learning_rate: 8.0000e-04
Epoch 3/100
1199/1200 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.5770 - mae: 2.3706
Epoch 3: val_loss did not improve from 2.74167
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 3.5770 - mae: 2.3706 - val_loss: 2.7519 - val_mae: 1.9105 - learning_rate: 8.0000e-04
Epoch 4/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.5566 - mae: 2.3584
Epoch 4: val_lo

In [222]:
def calculate_metrics(actual, predicted, name):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    print(f"\n===== {name} =====")
    print(f"MAE:  {mae:.3f} °C")
    print(f"RMSE: {rmse:.3f} °C")
    print(f"R²:   {r2:.4f}")

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

**FIX:** The next cell loads the checkpointed *best* model and computes `test_lstm_predictions` correctly. The original notebook had three further cells after this that recomputed `test_predicted_delta` / `test_lstm_predictions` / `test_persistence` using `model` (the last-epoch weights) instead of `best_model`, silently discarding the checkpointed result. Those duplicate cells have been removed so evaluation below always reflects the best checkpoint.

In [224]:
    # Load the actual best checkpoint explicitly
best_model = tf.keras.models.load_model(
    "best_cross_device_lstm.keras"
)

# Predict temperature change
test_predicted_delta = (
    best_model.predict(X_test)
    .flatten()
)

# Reconstruct future temperature
test_lstm_predictions = (
    test_current
    + test_predicted_delta
)

# Persistence baseline:
# future temperature = current temperature
test_persistence = test_current.copy()

315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [226]:
internal_baseline = calculate_metrics(

    test_actual,

    test_persistence,

    "Internal Test - Persistence"
)


===== Internal Test - Persistence =====
MAE:  2.402 °C
RMSE: 5.045 °C
R²:   0.8764


In [228]:

internal_lstm = calculate_metrics(
    test_actual,
    test_lstm_predictions,
    "Internal Test - LSTM"
)


===== Internal Test - LSTM =====
MAE:  2.283 °C
RMSE: 4.828 °C
R²:   0.8868


In [107]:
def evaluate_by_phase(

    actual,
    predicted,
    phases,
    model_name
):

    results = []


    for phase in np.unique(
        phases
    ):

        mask = (
            phases == phase
        )


        if mask.sum() < 10:

            continue


        mae = mean_absolute_error(

            actual[mask],

            predicted[mask]
        )


        rmse = np.sqrt(

            mean_squared_error(

                actual[mask],

                predicted[mask]
            )
        )


        results.append({

            "Model":
                model_name,

            "Phase":
                phase,

            "Samples":
                mask.sum(),

            "MAE":
                mae,

            "RMSE":
                rmse
        })


    return pd.DataFrame(
        results
    )

In [109]:
lstm_phase_results = evaluate_by_phase(

    test_actual,

    test_lstm_predictions,

    test_phases,

    "LSTM"
)

In [111]:
baseline_phase_results = evaluate_by_phase(

    test_actual,

    test_persistence,

    test_phases,

    "Persistence"
)

In [113]:
phase_comparison = pd.concat(

    [
        baseline_phase_results,
        lstm_phase_results
    ],

    ignore_index=True
)

display(
    phase_comparison
)

,Model,Phase,Samples,MAE,RMSE
0,Persistence,CHAOS,1598,3.540864,6.675724
1,Persistence,COOLING,357,2.601120,4.132945
2,Persistence,MIXED,1590,3.519057,6.517428
3,Persistence,OTHER,1069,2.245744,3.403431
4,Persistence,RAMP,1598,1.582040,3.320909
5,Persistence,TRANSITION,1594,1.881179,4.319024
6,LSTM,CHAOS,1598,3.340305,6.391042
7,LSTM,COOLING,357,2.474767,3.963509
8,LSTM,MIXED,1590,3.364098,6.375557
9,LSTM,OTHER,1069,2.329428,3.330311
